# Agente de Triagem de Currículos com Gemini
LLM-powered Resume Screening and ATS Optimization Agent.
######Developer: Fabiana S.S

In [1]:
!pip install google-genai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 5.5 MB/s eta 0:00:00


In [2]:
import os
import json
from google import genai
from google.genai import types
from google.colab import userdata
from google.colab import files
import pypdf
import re

In [3]:
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [4]:
uploaded = files.upload()


# O nome do arquivo enviado precisa ser extraído do dicionário retornado pelo upload.
caminho_do_pdf = list(uploaded.keys())[0]
print(f"Arquivo recebido: {caminho_do_pdf}")

Saving curriculo ficitício -Gabriel de Alonco.pdf to curriculo ficitício -Gabriel de Alonco.pdf
Arquivo recebido: curriculo ficitício -Gabriel de Alonco.pdf


In [5]:
leitor_pdf = pypdf.PdfReader(caminho_do_pdf)
curriculo_texto = ""
for pagina in leitor_pdf.pages:
    texto_pagina = pagina.extract_text()
    if texto_pagina:
        curriculo_texto += texto_pagina + "\n"

if not curriculo_texto.strip():
    raise ValueError("Não foi possível extrair texto do PDF. Verifique se o arquivo não é uma imagem escaneada.")

print(curriculo_texto[:500], "...")

 
Gabriel  Vasconcelos  de  Alonço  Cientista  de  Dados  Principal  &  Especialista  em  IA  Sênior  São  Paulo,  SP  |  +55  (11)  99999-0000  |  gabriel.alonco@emailficticio.com  |  linkedin.com/in/gabriel-alonco-ficticio  
Resumo  Profissional  Cientista  de  Computação  com  mais  de  8  anos  de  experiência  no  desenvolvimento,  
treinamento
 
e
 
implantação
 
de
 
modelos
 
de
 
Inteligência
 
Artificial
 
e
 
Aprendizado
 
de
 
Máquina
 
(Machine
 
Learning).
 
Especialista
 
em
 
Pro ...


## Upload da vaga em PDF
Em vez de colar o texto da vaga direto no notebook, agora ela também é enviada como anexo PDF, igual ao currículo — o texto é extraído automaticamente.

In [10]:
print("Faça o upload do PDF da vaga:")
uploaded_vaga = files.upload()
caminho_pdf_vaga = list(uploaded_vaga.keys())[0]
print(f"Arquivo da vaga recebido: {caminho_pdf_vaga}")

Faça o upload do PDF da vaga:


Saving crie um vaga fictícia para gerente de TI.pdf to crie um vaga fictícia para gerente de TI.pdf
Arquivo da vaga recebido: crie um vaga fictícia para gerente de TI.pdf


In [12]:
leitor_pdf_vaga = pypdf.PdfReader(caminho_pdf_vaga)
vaga_texto = ""
for pagina in leitor_pdf_vaga.pages:
    texto_pagina = pagina.extract_text()
    if texto_pagina:
        vaga_texto += texto_pagina + "\n"

if not vaga_texto.strip():
    raise ValueError("Não foi possível extrair texto do PDF da vaga. Verifique se o arquivo não é uma imagem escaneada.")

print(vaga_texto[:500], "...")

Vaga:  Gerente  de  Tecnologia  da  Informação  (TI)  Empresa:  NexaTech  Solutions  (Fictícia)  Modelo  de  Trabalho:  Híbrido  (São  Paulo  -  SP)  Regime  de  Contratação:  PJ  ou  CLT  
Sobre  a  Empresa  
A  NexaTech  é  uma  empresa  líder  em  inovação  digital  que  desenvolve  soluções  de  software  
escaláveis
 
para
 
o
 
setor
 
de
 
logística.
 
Estamos
 
em
 
um
 
momento
 
de
 
forte
 
expansão
 
e
 
buscamos
 
uma
 
liderança
 
que
 
não
 
apenas
 
gerencie
 
a
 
infraestrutura
 ...


## Mascaramento de PII (LGPD)
Antes de enviar o currículo ao Gemini, substituímos nome, e-mail, telefone e links de perfil por tokens (ex: `[NOME_CANDIDATO]`, `[EMAIL_1]`). O mapeamento fica só na sua máquina/sessão e é usado no final para devolver os dados reais — a API nunca recebe os dados originais.

In [13]:
def anonimizar_pii(texto):
    """Substitui dados pessoais identificáveis por tokens e retorna (texto_mascarado, mapa_pii).
    mapa_pii: dict token -> valor_real, usado depois para reidentificar a resposta do modelo."""
    mapa_pii = {}
    texto_mascarado = texto

    # E-mail
    emails = re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', texto_mascarado)
    for i, email in enumerate(dict.fromkeys(emails)):  # dict.fromkeys preserva ordem e remove duplicatas
        token = f"[EMAIL_{i+1}]"
        mapa_pii[token] = email
        texto_mascarado = texto_mascarado.replace(email, token)

    # Telefone (formatos BR comuns: +55 (11) 99999-0000, (11) 99999-0000, 11999990000 etc.)
    telefones = re.findall(r'\+?\d{1,3}\s?\(?\d{2,3}\)?\s?\d{4,5}-?\d{4}', texto_mascarado)
    for i, tel in enumerate(dict.fromkeys(telefones)):
        token = f"[TELEFONE_{i+1}]"
        mapa_pii[token] = tel
        texto_mascarado = texto_mascarado.replace(tel, token)

    # Links de perfil (LinkedIn, GitHub etc.)
    links = re.findall(r'(?:linkedin\.com|github\.com)/\S+', texto_mascarado)
    for i, link in enumerate(dict.fromkeys(links)):
        token = f"[LINK_PERFIL_{i+1}]"
        mapa_pii[token] = link
        texto_mascarado = texto_mascarado.replace(link, token)

    # Nome do candidato (heurística: primeira linha não vazia do currículo).
    # ATENÇÃO: heurística simples baseada na convenção comum de currículos (nome no topo).
    # Para produção com formatos variados, recomenda-se um NER dedicado (ex: spaCy pt_core_news)
    # em vez desta heurística posicional.
    linhas = texto_mascarado.strip().split("\n")
    if linhas and linhas[0].strip():
        nome_candidato = linhas[0].strip()
        token = "[NOME_CANDIDATO]"
        mapa_pii[token] = nome_candidato
        texto_mascarado = texto_mascarado.replace(nome_candidato, token)

    return texto_mascarado, mapa_pii


def reidentificar(objeto, mapa_pii):
    """Percorre recursivamente o JSON de resposta e troca os tokens pelos dados reais."""
    if isinstance(objeto, str):
        for token, valor_real in mapa_pii.items():
            objeto = objeto.replace(token, valor_real)
        return objeto
    elif isinstance(objeto, list):
        return [reidentificar(item, mapa_pii) for item in objeto]
    elif isinstance(objeto, dict):
        return {chave: reidentificar(valor, mapa_pii) for chave, valor in objeto.items()}
    else:
        return objeto

In [14]:
curriculo_texto_mascarado, mapa_pii = anonimizar_pii(curriculo_texto)

print("Tokens de PII detectados e mascarados:")
for token in mapa_pii:
    print(f"  {token}")

Tokens de PII detectados e mascarados:
  [EMAIL_1]
  [LINK_PERFIL_1]
  [NOME_CANDIDATO]


In [15]:
prompt_usuario = f"""
Aqui estão os insumos para análise:

[TEXTO DA VAGA]
{vaga_texto}

[CONTEÚDO DO CURRÍCULO EXTRAÍDO - COM TOKENS DE ANONIMIZAÇÃO]
{curriculo_texto_mascarado}
"""

In [16]:
system_instruction = """{
  "system_instruction": "Você é um Especialista em Recrutamento e Seleção (Tech Recruiter) sênior de nível internacional. Sua especialidade é analisar currículos, extrair dados com precisão cirúrgica e mapear competências reais contra descrições de vagas (Job Descriptions), fornecendo relatórios analíticos e adaptando a apresentação das informações sem nunca falsificar ou inventar dados.",
  "context": "O usuário fornecerá o texto de um currículo (extraído de um PDF, possivelmente com tokens de anonimização no lugar de dados pessoais) e o texto de uma vaga de emprego (também extraído de um PDF). O seu objetivo é processar esses dados através de um fluxo modular de trabalho para avaliar a aderência do candidato e gerar um currículo adaptado e otimizado para a vaga em questão.",
  "constraints": [
    "PROIBIDO INVENTAR OU ALUCINAR: Você não deve criar experiências, empresas, datas, projetos, tecnologias ou certificados que não estejam explicitamente mencionados no texto enviado.",
    "ADAPTAÇÃO BASEADA EM GROUNDING: Adaptar significa reordenar, mudar a ênfase de palavras-chave e reescrever bullet points para destacar impactos relevantes à vaga. Utilize os termos da vaga apenas quando houver equivalência real e comprovada no currículo fornecido.",
    "OMISSÃO ESTRATÉGICA: Se o currículo contiver experiências ou tecnologias totalmente irrelevantes para a vaga-alvo, reduza o espaço dedicado a elas, priorizando o que importa para o recrutador.",
    "IDIOMA: Se a vaga estiver em inglês e o currículo original em português, o novo currículo gerado e as análises devem ser traduzidos e entregues inteiramente em inglês.",
    "TOKENS DE ANONIMIZAÇÃO: O currículo fornecido pode conter tokens como [NOME_CANDIDATO], [EMAIL_1], [TELEFONE_1], [LINK_PERFIL_1] no lugar de dados pessoais reais. Esses tokens DEVEM ser reproduzidos exatamente como estão, sem traduzir, remover ou alterar seu formato — eles serão substituídos pelos dados reais em uma etapa posterior fora do modelo."
  ],
  "processo_de_trabalho": [
    "Etapa 1 - Job Analysis: Identifique as hard skills obrigatórias, soft skills desejadas e os termos-chave essenciais.",
    "Etapa 2 - Resume Analysis: Varra o histórico do candidato buscando experiências, tecnologias ou sinônimos que atendam aos requisitos mapeados na Etapa 1.",
    "Etapa 3 - Match Engine: Realize uma análise quantitativa ponderada (Hard Skills 50%, Experiência 25%, Soft Skills 15%, Certificações 10%), calculando a porcentagem estimada de match, separando competências encontradas e destacando as lacunas.",
    "Etapa 4 - Verification & Formatting: Redija o novo documento focando nas conquistas e tecnologias correlatas à vaga e execute uma checagem rigorosa de integridade antes de formular o JSON final."
  ],
  "verification_loop": [
    "Confirme: Nenhuma empresa externa foi adicionada ao currículo?",
    "Confirme: Nenhuma tecnologia ou ferramenta foi inventada?",
    "Confirme: Nenhuma certificação ou curso inexistente foi criado?",
    "Confirme: Todas as informações presentes no texto final possuem origem estrita e exclusiva no currículo enviado?"
  ],
  "few_shot_examples": [
    {
      "cenario_1": {
        "entrada": {
          "curriculo": "Experiência sólida na construção de painéis gerenciais utilizando Power BI para o setor financeiro.",
          "vaga": "Requisitos: Domínio em ferramentas de Business Intelligence (BI) e análise de indicadores."
        },
        "saida_esperada": {
          "competencias": "* Business Intelligence (Power BI)"
        }
      }
    },
    {
      "cenario_2": {
        "entrada": {
          "curriculo": "Conhecimento conceitual em computação em nuvem obtido através de estudos autônomos. Sem experiência prática.",
          "vaga": "Desejável: Certificação AWS Certified Cloud Practitioner ou AWS Solutions Architect."
        },
        "saida_esperada_antialucinacao": {
          "competencias": "* Fundamentos de Cloud Computing",
          "nota_de_validacao": "A certificação AWS não foi adicionada ao perfil adaptado por não constar como emitida no currículo original."
        }
      }
    }
  ],
  "output_format": {
    "type": "JSON",
    "schema": {
      "relatorio_aderencia": {
        "compatibilidade_estimada_porcentagem": "number",
        "hard_skills_encontradas": [
          "string"
        ],
        "soft_skills_encontradas": [
          "string"
        ],
        "lacunas_identificadas": [
          "string"
        ]
      },
      "curriculo_adaptado": {
        "nome_candidato": "string",
        "dados_contato": "string",
        "resumo_profissional": "string",
        "competencias_tecnicas": [
          "string"
        ],
        "experiencia_profissional": [
          {
            "cargo": "string",
            "empresa": "string",
            "periodo": "string",
            "conquistas_e_atividades": [
              "string"
            ]
          }
        ],
        "educacao_e_certificacoes": [
          "string"
        ]
      }
    }
  }
}"""

In [17]:


generation_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    temperature=0.2,  # Mantém a IA rígida às regras anti-alucinação
    response_mime_type="application/json"
)

# CORREÇÃO: 'gemini-2.5-flash' não está mais disponível para novos usuários da API
# (erro 404 NOT_FOUND). Usamos o alias 'gemini-flash-latest', que a própria Google
# mantém apontando para a versão estável mais recente do modelo Flash.
MODEL_NAME = "gemini-flash-latest"  # modelo definido explicitamente e usado de fato na chamada

In [18]:
print("Processando dados com o Gemini... Por favor, aguarde.")

try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_usuario,
        config=generation_config
    )
except Exception as e:
    raise RuntimeError(f"Falha ao chamar a API do Gemini: {e}")

print("Resposta recebida.")

Processando dados com o Gemini... Por favor, aguarde.
Resposta recebida.


In [19]:
try:
    resultado = json.loads(response.text)
except json.JSONDecodeError:
    print("⚠️ A resposta não veio em JSON válido. Texto bruto retornado:")
    print(response.text)
    resultado = None

In [20]:
# Os dados pessoais nunca saíram da sua máquina: o Gemini só viu os tokens.
# Aqui devolvemos nome/e-mail/telefone/link reais ao resultado final.
if resultado:
    resultado = reidentificar(resultado, mapa_pii)
    print(json.dumps(resultado, ensure_ascii=False, indent=2))

{
  "relatorio_aderencia": {
    "compatibilidade_estimada_porcentagem": 45,
    "hard_skills_encontradas": [
      "Arquitetura de Nuvem (AWS, GCP)",
      "Liderança Técnica de Equipes Multidisciplinares",
      "Tradução de Requisitos de Negócio em Soluções Técnicas"
    ],
    "soft_skills_encontradas": [
      "Comunicação assertiva (ponte entre técnico e negócios)",
      "Pensamento analítico",
      "Mentoria e desenvolvimento de pessoas"
    ],
    "lacunas_identificadas": [
      "Ausência de experiência com Governança de TI (ITIL, COBIT)",
      "Falta de vivência em Gestão de Orçamento (Capex/Opex) e Gestão de Fornecedores de TI",
      "Ausência de experiência com Segurança da Informação tradicional, DRP e conformidade LGPD",
      "Falta de experiência com migração de sistemas legados e implementação de ERP/CRM"
    ]
  },
  "curriculo_adaptado": {
    "nome_candidato": "Gabriel  Vasconcelos  de  Alonço  Cientista  de  Dados  Principal  &  Especialista  em  IA  Sênior  Sã

In [22]:
if resultado:
    with open("resultado_analise.json", "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)
    files.download("resultado_analise.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>